## Multicasting - Razpošiljanje 
- prevzeto uporablja **UDP**
- naslavljanje (razpošiljanje) omogoča dostavo skupinam ne glede na meje podomrežij 
Ideja: žemimo **POSLATI** podatke **VEČIM NASLOVNIKOM** 
Robna primera: 
- Mail:
    - Naštejemo prejemnike (**pošiljatelj upravlja z njimi**) (našteje naslove)
    - Prejemniki morajo poslati prošnjo za pridružitev (**Lahko časovno potratno**)
- Radisko oddajanje:
    - Prejemnik nastavi svojo postajo (multicasting) (nevemo kdo je naslovnik)
    - Antena oddaja signal (*brotcast*)
    + **Ni potrebno upravljanje z prejemniki** (manj dela)

**Izbira je načrtovalska:**
- Omrežje skrbi za prijavo, odjavo prejemnikov
- Sam administrator skrbi za prijavo odjavo

- Možnost oznake podatkov (Nekako jih moramo definirati)?
### Tehnologija
- **Register oznake podatkovnih tokov** (*Kako vemo da odstaja ARS*)
- Označiti moramo tok poratkov (npr. 17)
- Načrtovalska odločitev:
    - Obstaja [0-(256^k)-1], za k se moramo dogovoriti

- **Vse gre prek IP (od, komu, ...)** (komu - oznaka skupine (toka))
- Komu = (32 bit št.) --> 0 <= Naslov >= 2^32:
    - (IP:)
    - tip: ali govorimo o naslovu ali oznake skupine
    - naslov / oznaka

### Pridobitev skupine (Prijava multicast)  
Potrebujemo:    
- prijava na promet
- odjava
- poizvedba 
*Nekdo mora pripeljati promet v omrežje.*   

- Uporabnik pošlje usmerjevalniku, da bi se rad prijavil na skupino z tonom N
1. Usmerjevalnik pozna skupino in te prijavi
2. Usmerjevalnik ne pozna skupine poplavljanje (praša sosede, ...)
    
Poznamo dva protokola:  
- IGMP (Namenjen za IPv4)
- MLD -> je podoben z nekaj dodatnimi polji v paketu (Namenjen za IPv6)


## IGMP - Internet Group Management Protocol    
[IGMP_v1_v2_v3](./Ostalo_Ideje.ipynb)   
[IGMP_packet](./Slike/IGMP_packet.png)  
[IGMP_protocol](./Slike/IGMP_protocol.png)  
- Ne vključuje varnostnih mehanizmov ali integracijo
    - NI ponja za HMAC
    - Chech sum je namenjen samo za preverjanje bitov, še vedno lahko manipuliramo
    - RFC-ji eksplicitno ne zahtevajo avtentifikacije
- Delujen na omrežni plasti 
- Omogoča: prijavo, odjavo, zaznavanje drugih vmesnikov v skupini
Za prijavo in odjavo potrebuje poslati:   (kdo, kam)  
- **Svoj naslov**
- **ID skupine**

Postopek:
1. Pošlje zahtevo na usmerjevalnik (TTL nastavi na 1) (*da ne gre ven z omrežja*)
- **IP destination Address = ime skupine (ne IP)** *(zato da se znebimo težav z pošiljanjem)*
    - Usmerjevalnik pozna naslov te skupine:
        - Usmerjevalnik se prijavi v skupino
    - Usmerjevalnik poplavi omrežje: (ne pozna naslova (išče))
        - Poplavi: *ostale usmerjevalnike sprašuje če kdo ze prejema iz te skupine*
        - Če C **usmerjevalnik dobi dva paketa**: 
            - Originalni paket je tisti ki je prišel po **najkrajši poti**, drugi je kopija
                -  (Reverse Path Lookup)
                - (Najkrajšo pot pa ve tako da pogleda komu bi on poslal, da bi paket pišel do začetnega usmerjevalnika)

- Če se **ne odjaviš** skupina konstantno pošilja (*Ni tako kot pri DHCP protokolu*)
- IP **ni dejansko v paketu IGMP** (*dobimo z IP poizvedbe s katero pride*) 
Ukaz: type (prošnja/odjava)

2. **Odjavljamp se 224.0.0.2 (vsem usmerjevalnikom)** - da ostali ne posiljajo v bližino saj ni portebno
    - Če zelimo poizvedbo pošljemo IGMP Query na 224.0.0._ 
        - (Usmerjevalniki pošljejo IGMP Membership Report)
    - [Posebni_IPv4_naslovi](./Slike/PosebniIPv4.png)
    - [Posebni_IPv6_naslovi](./Omrezne_Plasti.ipynb)

## Membership ropport   
v IGMPv3    
[IGMPv3_Membership_ropport](./Slike/IGMP_MR.png)    

- Pošlje: Odjemalec (host) — npr. TV sprejemnik, računalnik, kamera, STB, IPTV box.
- Prejme: Lokalni usmerjevalnik
    - Usmerjevalnik potrebuje poročila, da ve:
        - na katerih portih mora poslati multicast promet
        - ali naj odstrani usmerjanje za določen kanal, če ni več gledalcev
        - ali naj izvede pruning v multicast drevesu
- S tem se prepreči, da multicast tok poplavlja celo omrežje.

Pošlje ga v dveh primerih:
- Ko se želi naročiti na multicast skupino
    -  IGMP Membership Report
- Kot odgovor na IGMP Query od usmerjevalnika
    - Pove: "Še vedno gledam kanal, ne odstrani me."


## Varnost
### Multicasting
- ali lahko paket razpošlje vsakdo ali samo član skupine?
- prisluškovalec se lahko naroči na poslušanje paketov in postane legitimni prejemnik
- zanesljivost:
-  kaj narediti, če samo en prejemnik javi, da ni dobil paketa?

## Prenos celih datotek
- Multicast podpira prenos celih datotek
- Zato ker deluje na UDP moramo uporabiti posebne protokole:
    - UFTP
    - MFTP